# 04 — M2 Threat Corpus Builder V2

**Goal:** Build a clean corpus of **threat** examples for the M2 four-class router.

**V2 Changes vs V1:**
1. `normalize_text()` applied to ALL text
2. **ImplicitHate REMOVED** (mislabeled — demographic hate ≠ threat)
3. **TRAC-3 threat REMOVED** (Bengali/Hindi, not English)
4. Smaller but CLEANER corpus (~4,356 rows)

**Sources kept:**
- UCB MHS (violence score ≥ 3.0)
- Hammer (direct threats)
- EDOS threats (category 1)
- Jigsaw (threat=1, keyword filtered)
- InViS (Violence=Yes, not sexual)
- ETHOS threat (threat_gap/)
- ConvAbuse threat (threat_gap/)
- AEGIS threat (majority-vote)

In [ ]:
# ============================================================
# CELL 1 — Setup + Shared Normalizer
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

BASE      = Path("/content/drive/MyDrive/AEGIS_ML")
RAW       = BASE / "data" / "raw"
THREAT    = RAW / "threat_gap"
PROCESSED = BASE / "data" / "processed" / "v5"
PROCESSED.mkdir(parents=True, exist_ok=True)

def normalize_text(text: str) -> str:
    """Canonical text normalization for AEGIS pipeline."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\brt\s+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else None

def text_hash(t):
    return hashlib.sha256(t.strip().encode('utf-8')).hexdigest()

print("✅ Setup complete")

In [ ]:
# ============================================================
# CELL 2 — Load All Threat Sources
# ============================================================

frames = []

print("="*70)
print("THREAT CORPUS V2 — LOADING SOURCES")
print("="*70)

# ── 1. UCB MHS — violence mean >= 3.0 ────────────────────────
print("\n[1] UCB MHS (pre-filtered)")
df = pd.read_csv(THREAT / "ucb_mhs_threat.csv")
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'threat'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows")

# ── 2. Hammer ─────────────────────────────────────────────────
print("\n[2] Hammer")
df = pd.read_csv(THREAT / "hammer_threat.csv")
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'threat'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows")

# ── 3. EDOS threats (category 1) ─────────────────────────────
print("\n[3] EDOS threats")
df = pd.read_csv(RAW / 'edos.csv')
df = df[df['label_category'] == '1. threats, plans to harm and incitement'].copy()
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'threat'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows")

# ── 4. Jigsaw — threat=1, keyword filtered ───────────────────
print("\n[4] Jigsaw (threat flag + keyword filter)")
df = pd.read_csv(RAW / 'jigsaw' / 'train.csv')
df = df[df['threat'] == 1].copy()
# Additional keyword filter to ensure threat quality
THREAT_KW = r'\b(kill|die|dead|murder|shoot|bomb|destroy|burn|stab|attack|hurt|threat|rape|hang|beat|execute|gun|knife|weapon|eliminate|slaughter)\b'
df = df[df['comment_text'].str.lower().str.contains(THREAT_KW, regex=True, na=False)].copy()
# Cap word count at 30 to avoid long forum rants
df['wc'] = df['comment_text'].str.split().str.len()
df = df[df['wc'] <= 30].copy()
df['text'] = df['comment_text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'threat'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows")

# ── 5. ETHOS threat ──────────────────────────────────────────
print("\n[5] ETHOS threat")
try:
    df = pd.read_csv(THREAT / "ethos_threat.csv")
    df['text'] = df['text'].apply(normalize_text)
    df = df[df['text'].notna() & (df['text'].str.len() > 0)]
    df['label'] = 'threat'
    frames.append(df[['text', 'label']].copy())
    print(f"  ✅ {len(df):,} rows")
except: print("  ⚠️ ETHOS file not found")

# ── 6. ConvAbuse threat ──────────────────────────────────────
print("\n[6] ConvAbuse threat")
try:
    df = pd.read_csv(THREAT / "convabuse_threat.csv")
    df['text'] = df['text'].apply(normalize_text)
    df = df[df['text'].notna() & (df['text'].str.len() > 0)]
    df['label'] = 'threat'
    frames.append(df[['text', 'label']].copy())
    print(f"  ✅ {len(df):,} rows")
except: print("  ⚠️ ConvAbuse threat file not found")

# ── 7. AEGIS Safety Corpus — majority-vote threat ────────────
print("\n[7] AEGIS Safety (threat labels)")
try:
    df = pd.read_csv(RAW / 'aegis_safety_corpus.csv')
    df = df[df['text_type'] == 'user_message'].copy()
    label_cols = ['labels_0','labels_1','labels_2','labels_3','labels_4']
    threat_labels = ['criminal planning/confessions', 'violence']
    
    def is_threat(row):
        votes = [str(row[c]).strip().lower() for c in label_cols if pd.notna(row[c])]
        if not votes: return False
        threat_votes = sum(1 for v in votes if v in threat_labels)
        return threat_votes > len(votes)/2
    
    df_threat = df[df.apply(is_threat, axis=1)].copy()
    df_threat['text'] = df_threat['text'].apply(normalize_text)
    df_threat = df_threat[df_threat['text'].notna() & (df_threat['text'].str.len() > 0)]
    df_threat['label'] = 'threat'
    frames.append(df_threat[['text', 'label']].copy())
    print(f"  ✅ {len(df_threat):,} rows")
except Exception as e:
    print(f"  ⚠️ AEGIS threat extraction failed: {e}")

# ══════════════════════════════════════════════════════════════
# ❌ REMOVED: ImplicitHate (mislabeled → discrimination)
# ❌ REMOVED: TRAC-3 threat (not English)
# ══════════════════════════════════════════════════════════════
print("\n❌ SKIPPED: ImplicitHate (moved to discrimination)")
print("❌ SKIPPED: TRAC-3 threat (Bengali/Hindi, not English)")

# ── COMBINE ──────────────────────────────────────────────────
print("\n" + "="*70)
print("COMBINING THREAT SOURCES")
print("="*70)

df_threat_all = pd.concat(frames, ignore_index=True)

# Dedup
before = len(df_threat_all)
df_threat_all['text_hash'] = df_threat_all['text'].apply(text_hash)
df_threat_all = df_threat_all.drop_duplicates(subset='text_hash', keep='first').copy()
df_threat_all = df_threat_all.drop(columns='text_hash')
print(f"  Dedup: {before:,} → {len(df_threat_all):,} ({before-len(df_threat_all):,} removed)")

# Filter min words
df_threat_all = df_threat_all[df_threat_all['text'].str.split().str.len() >= 3].copy()

print(f"\n  FINAL THREAT CORPUS: {len(df_threat_all):,} rows")

# ── Spot check ───────────────────────────────────────────────
print("\n📌 Random sample (10 examples):")
for _, row in df_threat_all.sample(10, random_state=42).iterrows():
    print(f"  • {row['text'][:100]}")

In [ ]:
# ============================================================
# CELL 3 — Save Threat Corpus
# ============================================================

print("="*70)
print("SAVING THREAT CORPUS V2")
print("="*70)

# Add metadata columns
df_threat_all['language'] = 'en'
df_threat_all['source_dataset'] = 'threat_v2'
df_threat_all['routing_method'] = 'manual'

out_path = PROCESSED / "m2_threat_v2.csv"
df_threat_all.to_csv(out_path, index=False)

size_kb = out_path.stat().st_size / 1024
print(f"\n  ✅ Saved: {out_path.name} ({size_kb:.1f} KB)")
print(f"  ✅ Rows: {len(df_threat_all):,}")

# Normalization validation
has_upper = df_threat_all['text'].str.contains(r'[A-Z]', regex=True).sum()
has_url   = df_threat_all['text'].str.contains(r'https?://', regex=True).sum()
print(f"\n  Validation:")
print(f"    Uppercase chars: {has_upper} (should be 0)")
print(f"    URLs remaining: {has_url} (should be 0)")
print(f"\n🎉 Threat V2 complete!")